**Prod figée :** `score_sortie` → Random Forest, `score_tele` → logistic (`PRODUCTION_MODELS`). Le ranking PR-AUC complet reste affiché (`meilleur_benchmark` peut différer du retenu). Graphique : `documentation/assets/comparaison_modeles_rf.png`.


## 0. Choix RF vs LightGBM / HistGB (et protocole de split)

Sur le **petit jeu CISIA**, la **forêt aléatoire** est figée en production (score sortie) — sklearn, SHAP, stabilité.
**LightGBM** et **HistGB** restent dans le benchmark : utiles pour la comparaison CIF ; souvent meilleurs à **grand volume**.
Si LightGBM gagne le PR-AUC de peu ou seulement le F2, on documente l’écart sans basculer la prod.

Contenu partagé avec la webapp : `/cisia/modeles` (`src.model.choix_algorithmes`).

In [ ]:
from pathlib import Path
import sys

from IPython.display import Markdown, display
import pandas as pd

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "Sujet.md").is_file() and ((p / "donnees" / "patients.csv").is_file() or (p / "patients.csv").is_file())
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.model.choix_algorithmes import contenu_choix_modeles

c = contenu_choix_modeles()
display(Markdown(f"### {c['titre']}"))
display(Markdown(c["contexte_n"]))
display(Markdown(
    f"**Décision actuelle :** {c['decision_actuelle']['libelle']} — "
    f"{c['decision_actuelle']['pourquoi']}"
))
display(Markdown(f"**HistGB (grand jeu) :** {c['histgb']['pour_grand_jeu']}"))

display(Markdown("#### Forêt — avantages"))
for a in c["rf"]["avantages"]:
    display(Markdown(f"- {a}"))
display(Markdown("#### Forêt — inconvénients"))
for a in c["rf"]["inconvenients"]:
    display(Markdown(f"- {a}"))
display(Markdown("#### HistGB — avantages"))
for a in c["histgb_detail"]["avantages"]:
    display(Markdown(f"- {a}"))
display(Markdown("#### HistGB — inconvénients"))
for a in c["histgb_detail"]["inconvenients"]:
    display(Markdown(f"- {a}"))

display(Markdown(f"### {c['protocole_split']['titre']}"))
display(Markdown(c["protocole_split"]["resume"]))
display(Markdown(c["protocole_split"]["pas_70_30_simple"]))

display(Markdown(f"### {c['unites_constantes']['titre']}"))
display(Markdown(c["unites_constantes"]["resume"]))
display(pd.DataFrame(c["unites_constantes"]["table"]))


c = contenu_choix_modeles()
display(Markdown(f"### {c['titre']}"))
display(Markdown(c["contexte_n"]))
display(Markdown(
    f"**Décision actuelle :** {c['decision_actuelle']['libelle']} — "
    f"{c['decision_actuelle']['pourquoi']}"
))
display(Markdown(f"**HistGB (grand jeu) :** {c['histgb']['pour_grand_jeu']}"))

display(Markdown("#### Forêt — avantages"))
for a in c["rf"]["avantages"]:
    display(Markdown(f"- {a}"))
display(Markdown("#### Forêt — inconvénients"))
for a in c["rf"]["inconvenients"]:
    display(Markdown(f"- {a}"))
display(Markdown("#### HistGB — avantages"))
for a in c["histgb_detail"]["avantages"]:
    display(Markdown(f"- {a}"))
display(Markdown("#### HistGB — inconvénients"))
for a in c["histgb_detail"]["inconvenients"]:
    display(Markdown(f"- {a}"))

display(Markdown(f"### {c['protocole_split']['titre']}"))
display(Markdown(c["protocole_split"]["resume"]))
display(Markdown(c["protocole_split"]["pas_70_30_simple"]))

display(Markdown(f"### {c['unites_constantes']['titre']}"))
display(Markdown(c["unites_constantes"]["resume"]))
import pandas as pd
display(pd.DataFrame(c["unites_constantes"]["table"]))


## 1. Imports & chemins


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "Sujet.md").is_file() and ((p / "donnees" / "patients.csv").is_file() or (p / "patients.csv").is_file())
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / "notebooks") not in sys.path:
    sys.path.insert(0, str(ROOT / "notebooks"))

from src.data.paths import ProjectPaths
from notebooks._utils.schema import FEATURES_CLES, SCHEMA_ATTENDU

paths = ProjectPaths(root=ROOT)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
print("Racine :", ROOT)
print("Curated :", paths.curated)
print("Models  :", paths.models)


## 2. Chargement des features curated


In [ ]:
p_so = paths.curated / "features_score_sortie.parquet"
p_te = paths.curated / "features_score_tele.parquet"
if not p_so.is_file() or not p_te.is_file():
    raise FileNotFoundError(
        f"Parquets manquants.\nAttendu :\n- {p_so}\n- {p_te}\n"
        "Exécutez 03_preparation_modele.ipynb ou : PYTHONPATH=. python scripts/run_pipeline.py"
    )

features_sortie = pd.read_parquet(p_so)
features_tele = pd.read_parquet(p_te)

for nom, df in [("sortie", features_sortie), ("tele", features_tele)]:
    manquantes = [c for c in FEATURES_CLES if c not in df.columns]
    if manquantes:
        raise AssertionError(f"{nom}: colonnes manquantes {manquantes}")
    print(f"✓ features_score_{nom}: {df.shape}")


## 3. Paramètres d’entraînement


In [ ]:
from src.model.deploy import train_score

# True = Optuna complet (spec jury). False = catalogue rapide (smoke).
USE_OPTUNA = True
OPTUNA_TRIALS = 25
# Si False et que les bundles existent, on réutilise sans ré-entraîner.
FORCE_RETRAIN = True

print("USE_OPTUNA =", USE_OPTUNA, "| trials =", OPTUNA_TRIALS, "| FORCE_RETRAIN =", FORCE_RETRAIN)


## 4. Entraînement score_sortie


In [ ]:
bundle_so = paths.models / "score_sortie_bundle.joblib"
metrics_so_path = paths.models / "score_sortie_metrics.json"

if not FORCE_RETRAIN and bundle_so.is_file() and metrics_so_path.is_file():
    import json
    metrics_sortie = json.loads(metrics_so_path.read_text(encoding="utf-8"))
    print("⏭ Bundle sortie déjà présent — skip (FORCE_RETRAIN=False)")
else:
    print("▶ train_score score_sortie (Optuna=%s)…" % USE_OPTUNA)
    metrics_sortie = train_score(
        features_sortie,
        score_name="score_sortie",
        paths=paths,
        use_optuna=USE_OPTUNA,
        optuna_trials=OPTUNA_TRIALS,
    )
    print("✓ score_sortie retenu (prod) :", metrics_sortie.get("retenu"),
      "| meilleur PR-AUC benchmark :", metrics_sortie.get("meilleur_benchmark"))

display(pd.DataFrame([{
    "score": "sortie",
    "retenu_prod": metrics_sortie.get("retenu"),
    "meilleur_benchmark": metrics_sortie.get("meilleur_benchmark"),
    "ranking": " → ".join(metrics_sortie.get("ranking_test_pr_auc", [])[:4]),
    "n_train": metrics_sortie.get("n_train"),
    "n_test": metrics_sortie.get("n_test"),
    "optuna": metrics_sortie.get("optuna"),
}]))


## 5. Entraînement score_tele


In [ ]:
bundle_te = paths.models / "score_tele_bundle.joblib"
metrics_te_path = paths.models / "score_tele_metrics.json"

if not FORCE_RETRAIN and bundle_te.is_file() and metrics_te_path.is_file():
    import json
    metrics_tele = json.loads(metrics_te_path.read_text(encoding="utf-8"))
    print("⏭ Bundle télé déjà présent — skip (FORCE_RETRAIN=False)")
else:
    print("▶ train_score score_tele (Optuna=%s)…" % USE_OPTUNA)
    metrics_tele = train_score(
        features_tele,
        score_name="score_tele",
        paths=paths,
        use_optuna=USE_OPTUNA,
        optuna_trials=OPTUNA_TRIALS,
    )
    print("✓ score_tele retenu (prod) :", metrics_tele.get("retenu"),
      "| meilleur PR-AUC benchmark :", metrics_tele.get("meilleur_benchmark"))

display(pd.DataFrame([{
    "score": "tele",
    "retenu_prod": metrics_tele.get("retenu"),
    "meilleur_benchmark": metrics_tele.get("meilleur_benchmark"),
    "ranking": " → ".join(metrics_tele.get("ranking_test_pr_auc", [])[:4]),
    "n_train": metrics_tele.get("n_train"),
    "n_test": metrics_tele.get("n_test"),
    "optuna": metrics_tele.get("optuna"),
}]))


## 6. Tableau comparatif + artefacts artefacts


In [ ]:
import json

def resume_metrics(path: Path) -> dict:
    m = json.loads(path.read_text(encoding="utf-8"))
    retenu = m["retenu"]
    test = m.get(retenu, {}).get("test", {})
    return {
        "score": m.get("score"),
        "retenu": retenu,
        "pr_auc_test": test.get("pr_auc"),
        "roc_auc_test": test.get("roc_auc"),
        "f2_test": test.get("f2"),
        "seuil": test.get("threshold"),
        "optuna": m.get("optuna"),
    }

metrics_so_path = paths.models / "score_sortie_metrics.json"
metrics_te_path = paths.models / "score_tele_metrics.json"
comparatif = pd.DataFrame([resume_metrics(metrics_so_path), resume_metrics(metrics_te_path)])
display(comparatif)

print("Artefacts :")
for p in [
    paths.models / "score_sortie_bundle.joblib",
    paths.models / "score_sortie_metrics.json",
    paths.models / "score_tele_bundle.joblib",
    paths.models / "score_tele_metrics.json",
]:
    print(("✓" if p.is_file() else "✗"), p)

print("✓ Entraînement terminé — enchaînez avec 05_lancement_modele.ipynb")


## Suite du parcours

**Prochaine étape :** [05 — Lancement / inférence](05_lancement_modele.ipynb)

| Étape | Notebook |
|-------|----------|
| → **05 — Lancement / inférence** | [`05_lancement_modele.ipynb`](05_lancement_modele.ipynb) |
| Benchmark détaillé (complément) | [`01_benchmark_optuna.ipynb`](01_benchmark_optuna.ipynb) |
| Sommaire | [`../00_guide/00_sommaire.ipynb`](../00_guide/00_sommaire.ipynb) |
